# 5.1 Class area summary (km² and % cover)

Counts the pixels of each typology class in every Time Interval raster (`TimeInterval*.tif`)
and in the newest Time Series raster, converts them to km² and % cover, and writes one CSV
with all rasters side by side.

- Projection: South America Albers Equal Area Conic (ESRI:102033)
- Cell size: 31.887054 m, so one pixel = **1016.79 m²** (not 900 m²)
- Area = pixel count × 1016.79 m²; % cover = class area / total classified area of that raster

**Packages** (all already in `environment.yml`): `numpy`, `pandas`, `rasterio`

In [ ]:
### Packages ###
import time
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

In [ ]:
### Parameters ###
TYP = Path(r"./data/5_Typology")                                   # folder with the typology rasters
TS_PATH = TYP / "combined_rc_timeseries" / "combined_rc.tif"       # newest Time Series typology
OUT_CSV = TYP / "class_area_km2_summary.csv"

PX_M2 = 1016.79        # m2 per pixel (31.887054 m x 31.887054 m, ESRI:102033)
ROWS = 128             # rows read at a time; lower if RAM is tight

CLASSES = {1: "Attrition", 2: "Aggregation", 3: "Shrinkage", 4: "Perforation", 5: "Deformation",
           6: "Persistent", 7: "Enlargement", 8: "Dissection", 9: "Fragmentation per se", 10: "Creation"}

In [ ]:
### Find the rasters ###
RASTERS = {p.stem: p for p in sorted(TYP.glob("TimeInterval*.tif"))}
RASTERS["TimeSeries 1990-2020"] = TS_PATH

print("rasters to summarize:")
for name, p in RASTERS.items():
    print("  ", "OK     " if p.exists() else "MISSING", name, "->", p)
assert all(p.exists() for p in RASTERS.values()), "a raster is missing; check the paths above"

In [ ]:
### Functions ###
def check_raster(path):
    """Print the CRS and pixel size, and warn if the pixel area is not PX_M2."""
    with rasterio.open(path) as src:
        px = abs(src.transform.a * src.transform.e)
        print("  CRS:", src.crs, "| cell size:", src.res, "| pixel m2:", round(px, 2))
        if abs(px - PX_M2) > 0.01:
            print("  WARNING: pixel area is", round(px, 2), "m2, not", PX_M2)


def count_classes(path):
    """Pixel count per class value (NoData excluded), read in strips."""
    counts = np.zeros(256, np.int64)
    with rasterio.open(path) as src:
        for r in range(0, src.height, ROWS):
            w = Window(0, r, src.width, min(ROWS, src.height - r))
            a = src.read(1, window=w, masked=True)
            v = np.ma.getdata(a)[~np.ma.getmaskarray(a)].astype(np.int64)
            counts += np.bincount(v[(v >= 0) & (v < 256)], minlength=256)
    other = int(counts.sum() - sum(counts[k] for k in CLASSES))
    if other:
        print("  WARNING:", f"{other:,}", "pixels have values that are not classes 1-10")
    return {k: int(counts[k]) for k in CLASSES}


def class_table(counts):
    """Pixels, km2 and % cover per class, plus a TOTAL row."""
    df = pd.DataFrame({"Class": list(CLASSES.values()), "pixels": list(counts.values())})
    total = df["pixels"].sum()
    df["km2"] = df["pixels"] * PX_M2 / 1e6
    df["%cover"] = 100 * df["pixels"] / total
    df.loc[len(df)] = ["TOTAL", total, total * PX_M2 / 1e6, 100.0]
    return df

In [ ]:
### Run: count every raster ###
tables = {}
for name, p in RASTERS.items():
    t0 = time.time()
    print(name)
    check_raster(p)
    tables[name] = class_table(count_classes(p))
    print(f"  done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
### Combine into one table and save the CSV ###
combined = pd.DataFrame({"Class": list(CLASSES.values()) + ["TOTAL"]})
for name, t in tables.items():
    combined[f"{name} pixels"] = t["pixels"].values
    combined[f"{name} km2"] = t["km2"].round(1).values
    combined[f"{name} %cover"] = t["%cover"].round(4).values

combined.to_csv(OUT_CSV, index=False)
print("saved", OUT_CSV)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
combined